In [ ]:
import pandas as pd 
import os
# Paths
DOWNLOAD_DIR = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
file_path = "companies.csv"  # Update with the actual path to your CSV file

# Load CIK-to-Ticker mapping from CSV
companies_df = pd.read_csv(file_path, header=None)
cik_dict = {str(row[0]).zfill(10): row[1] for _, row in companies_df.iloc[1:].iterrows()}  # CIK -> Ticker

# Rename CIK folders to Ticker names
for cik_folder in os.listdir(DOWNLOAD_DIR):
    cik_path = os.path.join(DOWNLOAD_DIR, cik_folder)

    # Ensure it's a directory and a valid CIK
    if os.path.isdir(cik_path) and cik_folder in cik_dict:
        ticker_name = cik_dict[cik_folder]  # Get corresponding ticker
        new_path = os.path.join(DOWNLOAD_DIR, ticker_name)

        if not os.path.exists(new_path):  # Avoid overwriting existing folders
            os.rename(cik_path, new_path)
            print(f"✅ Renamed {cik_folder} ➝ {ticker_name}")
        else:
            print(f"⚠️ Skipped {cik_folder} (Ticker {ticker_name} already exists!)")
    else:
        print(f"❌ Skipped {cik_folder} (No matching ticker found)")


In [ ]:
import os
import pandas as pd
import time
import datetime
from tqdm import tqdm
from sec_edgar_downloader import Downloader
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import local

# Constants
DOWNLOAD_DIR = "sec-edgar-filings"
FORM = "10-K"
YEARS = range(2015, 2025)
MAX_THREADS = 10
EMAIL = "hitiras@hotmail.com"

# Thread-local downloader
thread_local = local()

def get_downloader():
    if not hasattr(thread_local, "dl"):
        thread_local.dl = Downloader(DOWNLOAD_DIR, EMAIL)
    return thread_local.dl

# Load tickers and CIKs
file_path = "companies.csv"
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[0:10]  # skip header row
tickers = companies_df.iloc[:, 1].tolist()
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iterrows()}

# Download for one ticker
def download_filings(cik, ticker):
    dl = get_downloader()
    try:
        start = datetime.datetime.now()
        print(f"[{start.strftime('%H:%M:%S')}] 📥 {ticker} - Starting download...")
        #time.sleep(1.5)
        dl.get(FORM, cik, after=f"{YEARS[0]}-01-01", before=f"{YEARS[-1]}-12-31")
        end = datetime.datetime.now()
        print(f"[{end.strftime('%H:%M:%S')}] ✅ {ticker} - Done in {end - start}")
    except Exception as e:
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] ❌ {ticker} - Error: {e}")
        time.sleep(10)

# Main
def main():
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {
            executor.submit(download_filings, cik_dict[ticker], ticker): ticker
            for ticker in tickers
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading 10-Qs"):
            ticker = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] ❌ Thread failed for {ticker}: {e}")
                time.sleep(10)

if __name__ == "__main__":
    print("🚀 Starting optimized download with 1 Downloader per thread...\n")
    main()
